# IDX — Tiered Cold Archive: Operator Validation (Colab)

**Bukan trading runtime.** Hanya storage plane:

```
GitHub Tier 1 → Colab bridge → Drive pool (drive_01…N)
```

Invariant:
- LIVE_EXECUTION = FALSE
- BROKER_EXECUTION = FALSE
- Tier-1 → Drive: **copy-forward**, tidak auto-delete GitHub
- Hapus dari tier hanya setelah salinan di tier lain **terverifikasi** (SHA-256 + size)


In [ ]:
# ── 0) Config (edit jika perlu) ─────────────────────────────────────────
REPO_URL = "https://github.com/whatman42/idx.git"
REPO_DIR = "/content/idx"
BRANCH = "main"

DRIVE_POOL = {
    "drive_01": "/content/drive/MyDrive",
    # "drive_02": "/content/drive_acct2/MyDrive",
}

MAX_TIER1 = 20
MIN_RECOVERY = 3
SOURCE_STAGING = "/content/staging_archives"


In [ ]:
# ── 1) Mount Google Drive ───────────────────────────────────────────────
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("Mounted /content/drive")
except Exception as e:
    print("Colab drive.mount failed or not Colab:", e)
    Path("/content/drive/MyDrive").mkdir(parents=True, exist_ok=True)

for sid, root in DRIVE_POOL.items():
    Path(root).mkdir(parents=True, exist_ok=True)
    print(f"  pool {sid} -> {root} exists={Path(root).exists()}")


In [ ]:
# ── 2) Clone / update repo ──────────────────────────────────────────────
import subprocess, sys
from pathlib import Path

def sh(cmd):
    print("+", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError(cmd)
    if r.stdout.strip():
        print(r.stdout[-1500:])
    return r

if not Path(REPO_DIR, ".git").exists():
    sh(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}")
else:
    sh(f"git -C {REPO_DIR} fetch origin {BRANCH} && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull --ff-only origin {BRANCH}")

sys.path.insert(0, REPO_DIR)
print("HEAD:", open(f"{REPO_DIR}/.git/HEAD").read().strip())
import src.python.archive as _a
print("archive package OK")


In [ ]:
# ── 3) Build Drive pool + engines ───────────────────────────────────────
from pathlib import Path
from src.python.archive.drive_fs_backend import DriveFsBackend
from src.python.archive.tiers import DrivePool, TierLifecycleEngine, TierPromotionPolicy
from src.python.archive.tier_plan import plan_tier1_promotions, operational_archive_status
from src.python.archive.migration import MigrationEngine, discover_local_sources
from src.python.archive.integrity import sha256_bytes

pool = DrivePool()
for sid, root in DRIVE_POOL.items():
    pool.register(sid, root, account_alias=sid, capacity_limit_bytes=0, priority=10 + len(pool.backends))

journal = Path(list(DRIVE_POOL.values())[0]) / "IDX" / "cold_archive" / "migration_journal.json"
eng = TierLifecycleEngine(
    pool=pool,
    journal_path=journal,
    policy=TierPromotionPolicy(max_tier1_archives=MAX_TIER1, min_recovery_archives=MIN_RECOVERY),
)
print(operational_archive_status(pool=pool))
print("journal:", journal)


In [ ]:
# ── 4) GitHub Tier-1 inventory (Release assets) ─────────────────────────
import os, json, urllib.request
from pathlib import Path

Path(SOURCE_STAGING).mkdir(parents=True, exist_ok=True)

def list_github_release_assets(owner="whatman42", repo="idx", token=None):
    url = f"https://api.github.com/repos/{owner}/{repo}/releases?per_page=20"
    req = urllib.request.Request(url, headers={"Accept": "application/vnd.github+json", "User-Agent": "idx-colab"})
    if token:
        req.add_header("Authorization", f"Bearer {token}")
    with urllib.request.urlopen(req, timeout=60) as resp:
        releases = json.loads(resp.read().decode())
    assets = []
    for rel in releases:
        for a in rel.get("assets") or []:
            name = a.get("name") or ""
            if name.endswith(".tar.gz") or name.endswith(".tgz"):
                assets.append({
                    "archive_id": Path(name).stem,
                    "filename": name,
                    "size_bytes": int(a.get("size") or 0),
                    "download_url": a.get("browser_download_url"),
                    "created_at": a.get("created_at") or rel.get("published_at") or "",
                    "release_tag": rel.get("tag_name") or "",
                    "source": "github_release",
                })
    return assets

GH_TOKEN = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN")
try:
    gh_assets = list_github_release_assets(token=GH_TOKEN)
except Exception as e:
    print("GitHub releases list failed:", e)
    gh_assets = []

print(f"GitHub release .tar.gz assets: {len(gh_assets)}")
for a in gh_assets[:10]:
    print(" ", a["filename"], a["size_bytes"], a.get("release_tag"))

staged = discover_local_sources(Path(SOURCE_STAGING))
print(f"Already staged locally: {len(staged)}")


In [ ]:
# ── 5) Download eligible assets into staging ────────────────────────────
from pathlib import Path
import urllib.request

def download_asset(url: str, dest: Path, token=None):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print("skip exists", dest.name)
        return dest
    req = urllib.request.Request(url, headers={"User-Agent": "idx-colab"})
    if token:
        req.add_header("Authorization", f"Bearer {token}")
    with urllib.request.urlopen(req, timeout=300) as resp:
        data = resp.read()
    dest.write_bytes(data)
    print("downloaded", dest.name, len(data), "sha", sha256_bytes(data)[:12])
    return dest

MAX_DOWNLOAD = 5
for a in gh_assets[:MAX_DOWNLOAD]:
    url = a.get("download_url")
    if not url:
        continue
    dest = Path(SOURCE_STAGING) / a["filename"]
    try:
        download_asset(url, dest, token=GH_TOKEN)
    except Exception as e:
        print("download fail", a["filename"], e)

tier1 = discover_local_sources(Path(SOURCE_STAGING))
print("Tier-1 staging inventory:", len(tier1))
plan = plan_tier1_promotions(tier1, pool=pool, max_tier1=MAX_TIER1)
print(json.dumps({k: plan[k] for k in plan if k != "promote_plan"}, indent=2))
print("promote_plan sample:", plan["promote_plan"][:5])


In [ ]:
# ── 6) Promote staging → Drive pool (SHA-256 dual verify) ───────────────
from pathlib import Path
import json

results = []
for s in tier1:
    data = Path(s["path"]).read_bytes()
    rec = eng.promote_to_drive(
        data,
        archive_id=s["archive_id"],
        filename=s["filename"],
        cycle_date=(s.get("created_at") or "")[:10],
        source="github_staging",
    )
    results.append(rec)
    print(rec["archive_id"], rec["status"], rec.get("error_code"), "target=", rec.get("target_storage_id"), "del_src=", rec.get("deleted_source"))

ok = sum(1 for r in results if r["status"] in ("MIGRATED", "ARCHIVE_ALREADY_MIGRATED"))
fail = sum(1 for r in results if r["status"] == "FAILED")
conflict = sum(1 for r in results if r["status"] == "CONFLICT")
print(f"\nPROMOTE summary: ok={ok} fail={fail} conflict={conflict} total={len(results)}")
assert all(r.get("deleted_source") is False for r in results), "Tier-1 source must never auto-delete"


In [ ]:
# ── 7) Recovery verification (sample) ──────────────────────────────────
import random

locs = pool.list_all_locations()
verified = [L for L in locs if L.verified]
print("verified drive copies:", len(verified))
sample = verified if len(verified) <= 5 else random.sample(verified, 5)

recovery = []
for L in sample:
    be = pool.backends[L.storage_id]
    mig = MigrationEngine(drive=be, journal_path=journal)
    out = mig.restore_verify(L.archive_id)
    recovery.append({"archive_id": L.archive_id, "storage_id": L.storage_id, **out})
    print(L.archive_id, L.storage_id, out)

bad = [r for r in recovery if not r.get("ok")]
print("recovery failures:", len(bad))
if bad:
    print("FAIL CLOSED — do not release any source tier")
else:
    print("recovery sample PASS")


In [ ]:
# ── 8) Drive → Drive capacity rotation (optional) ───────────────────────
if len(pool.backends) < 2:
    print("Skip Drive→Drive: need ≥2 pool members. Register drive_02 di Config.")
else:
    locs = sorted(pool.list_all_locations(), key=lambda L: (L.created_at, L.archive_id))
    if not locs:
        print("No archives to rotate")
    else:
        oldest = locs[0]
        print("rotate", oldest.archive_id, "from", oldest.storage_id)
        r = eng.promote_between_drives(oldest.archive_id, from_storage_id=oldest.storage_id)
        print(r)


In [ ]:
# ── 9) Final operator report ────────────────────────────────────────────
import json
from datetime import datetime, timezone
from pathlib import Path

report = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "repo": REPO_URL,
    "branch": BRANCH,
    "operational_status": operational_archive_status(pool=pool),
    "tier1_staged": len(tier1),
    "drive_locations": len(pool.list_all_locations()),
    "verified_drive_copies": sum(1 for L in pool.list_all_locations() if L.verified),
    "promote_results": results,
    "recovery_sample": recovery,
    "claims": {
        "software_lifecycle_ci": "certified_separately",
        "real_drive_transfer_this_run": ok > 0,
        "github_tier1_deleted": False,
        "live_execution": False,
        "broker_execution": False,
    },
}
out = Path(list(DRIVE_POOL.values())[0]) / "IDX" / "cold_archive" / "operator_validation_report.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(report, indent=2, default=str))
print("Wrote", out)
print(json.dumps(report["claims"], indent=2))
print(json.dumps(report["operational_status"], indent=2))
